In [ ]:
import sqlite3
import pandas as pd
from sentence_transformers import SentenceTransformer
import numpy as np
from hashlib import sha256

In [88]:
def vectorisation_hobbies():
    hobbys_csv_path = 'ChatGPT/liste_hobbies.csv'

    hobbies_df = pd.read_csv(hobbys_csv_path)
    hobbies = hobbies_df['Hobby'].tolist()
    
    model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
    vectors = model.encode(hobbies)

    return hobbies, vectors

In [89]:
def vectorisation_traits():
    traits_csv_path = 'ChatGPT/liste_traits_caractere.csv'

    traits_df = pd.read_csv(traits_csv_path)
    traits = traits_df['Trait'].tolist()
    
    model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
    vectors = model.encode(traits)

    return traits, vectors

In [90]:
def vectorisation_metiers():
    metiers_csv_path = 'ChatGPT/liste_metiers.csv'

    metiers_df = pd.read_csv(metiers_csv_path)
    metiers = metiers_df['Metier'].tolist()
    
    model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
    vectors = model.encode(metiers)

    return metiers, vectors

In [91]:
def load_Users_DB(cursor: sqlite3.Cursor, nbr_users: int):
    df_users = pd.read_json('Fake_profiles/fake_profiles_1_000_000.json', orient="records", encoding='utf-8')
    df_users = df_users.sample(n=nbr_users, random_state=42).reset_index(drop=True)
    
    # Id,Prénom,Nom,Sexe,Âge,Ville,Hobby,Trait,Job
    for index, row in df_users.iterrows():
        # Insertion dans la table Users
        cursor.execute(
            """
            INSERT INTO Users (user_id, email, password_hash, nom, prenom, age, genre)
            VALUES (?, ?, ?, ?, ?, ?, ?)
            """,
            (row['Id'], row['Prénom'].lower() + "."  + row['Nom'].lower() + "@mail.com", sha256(row['Nom'].encode()).hexdigest(), row['Nom'], row['Prénom'], row['Âge'], row['Sexe'])
        )
        
        # Insertion dans la table Profil_Embeddings
        tab_embeddings = row['Hobby'] + row['Trait']
        tab_embeddings.append(row['Job'])
        
        for item in tab_embeddings:
            cursor.execute(
                """
                SELECT embeddings_id FROM Embeddings
                WHERE text_initial = ?
                """,
                (item,)
            )
            item_id = cursor.fetchone()[0]
            cursor.execute(
                """
                INSERT INTO Profil_Embeddings (user_id, embeddings_id)
                VALUES (?, ?)
                """,
                (row['Id'], item_id)
            )

In [92]:
def load_sql():
    """
    """
    # Connexion à la base de données SQLite
    conn = sqlite3.connect('Y-love.db')  # Se connecter à la base de données
    cur = conn.cursor()  # Créer un curseur

    try:
        with open("schema.sql", 'r') as fichier:
            schema_sql = fichier.read()
        cur.executescript(schema_sql)  # execution du script
        print("Script schema SQL exécuté avec succès")
        
        # Insertion des vecteurs de hobbies
        hobbies, vectors_hobbies = vectorisation_hobbies()
        for i, hobby in enumerate(hobbies):
            cur.execute(
                "INSERT INTO Embeddings (model_version, categorie, text_initial, vecteur) VALUES (?, ?, ?, ?)",
                ("paraphrase-multilingual-MiniLM-L12-v2", "hobby", hobby, vectors_hobbies[i])
            )
        print("Insertion des vecteurs de hobbies effectuée")
        
        # Insertion des vecteurs de traits
        traits, vectors_traits = vectorisation_traits()
        for i, trait in enumerate(traits):
            cur.execute(
                "INSERT INTO Embeddings (model_version, categorie, text_initial, vecteur) VALUES (?, ?, ?, ?)",
                ("paraphrase-multilingual-MiniLM-L12-v2", "trait", trait, vectors_traits[i])
            )
        print("Insertion des vecteurs de traits effectuée")
        
        # Insertion des vecteurs de métiers
        metiers, vectors_metiers = vectorisation_metiers()
        for i, metier in enumerate(metiers):
            cur.execute(
                "INSERT INTO Embeddings (model_version, categorie, text_initial, vecteur) VALUES (?, ?, ?, ?)",
                ("paraphrase-multilingual-MiniLM-L12-v2", "metier", metier, vectors_metiers[i])
            )
        print("Insertion des vecteurs de métiers effectuée")
        
        # Insertion des faux utilisateurs
        load_Users_DB(cur, 10_000)
        print("Insertion des faux utilisateurs effectuée")
        
        # Insertion des 'vrai' utilisateurs
        with open("data.sql", 'r') as fichier:
            data_sql = fichier.read()
        cur.executescript(data_sql)  # execution du script
        print("Script data SQL exécuté avec succès")
        
        conn.commit()  # Valider les changements
        print("Données insérées avec succès")
        conn.close()

    except sqlite3.Error as e:
        conn.rollback()  # Annuler les changements en cas d'erreur
        conn.close()
        print(f"Une erreur s'est produite : {e}")
    
    conn.close()

In [93]:
load_sql()
# Temps : 1m

Script schema SQL exécuté avec succès
Insertion des vecteurs de hobbies effectuée
Insertion des vecteurs de traits effectuée
Insertion des vecteurs de métiers effectuée
Insertion des faux utilisateurs effectuée
Script data SQL exécuté avec succès
Données insérées avec succès


## Verification du stockage des données
---

Surtout pour vérifier le stockage des vecteurs

In [94]:
# Connexion à la base de données SQLite
conn = sqlite3.connect('Y-love.db')  # Se connecter à la base de données
cur = conn.cursor()  # Créer un curseur

cur.execute("SELECT * FROM Embeddings")
rows = cur.fetchall()
print(rows[0])  # Affiche la première ligne de la table Embeddings
print()
print(rows[0][2], ":", rows[0][3], "->", np.frombuffer(rows[0][4], dtype=np.float32))
# np.frombuffer(rows[0][3], dtype=np.float32) -> pour convertir le blob (str de bits) en tableau numpy
conn.close()

(1, 'paraphrase-multilingual-MiniLM-L12-v2', 'hobby', 'Football', b'\x90\x07\xdf\xbc\x8eM#>\xb4\xc8\xf3\xbe\xb3X\xd8\xbe1\xe2\xb3>\x01\x93<>7\xed\xd2>\xb9\x08\x88>\xaby4>\xdf\xa1:>\x8a^\x17\xbf\xec\x01\xdc\xbew\x8cm>\xc6\xd3\x89>\xe9+&\xbeE\xbf\xa6\xbe\x83\xbbh\xbe\xac\xae\x88\xbe\x19\x8eO>\xe0nv\xbeH\xf7\xc4>)v\x94>\xf4\xe1H\xbe\xcc\xec+\xbf4VD\xbe\x1b4|\xbeW\xdb\xb8=\xc8\x94\xca<#\xf1\xb7\xbe\xe9*\x18\xbf\xa9\xa4\xee>\xc9\xcd\xc0\xbd\x89\x02\xb9>\x00\x1b<>\x0f\x08g\xbe\x8c+\xd6=\x9d!\xc8=u-\xad\xbe\x08\x89\xde>1t\x04?nt\x16\xbeh\xc9=\xbe\t\x89<={fI=+\x16<>\xe6\x04\x9d>\xdbF\x01\xbd@[\xad= *\xe3>\'\xc5\xc4>\x81\xa8\x91>\x05\x82#?y\xae\t\xbf\xa9\x0e\x1e=\x05C\xf2=5\x7f\xfc>E\xdfn>\xda\xa2\xa2>\x0c\x95\x0e\xbf\x1d\xba\xdd>\x91\xae\xa2>s%&>\x0c\xee\xc5\xbe\x81U\x81=\xed\xc5\x15\xbf\x85\'\x04\xbf\xbb{\xa9\xbc+W%\xbc4\x1cr\xbeX\xe3&>\x01\xffB\xbe\xe0\xa9.\xbc\x99\xdc\x0f>\xe5\xddp>5\x1b\x9f>c\x13\xde\xbd\x04\x8f\x8e>\x95\xaf\xa8=\xe1/\xad>\x85\xc48\xbe{\x91\x94\xbe$M\xfd=@\x1cA\xbd@c\x88\x

In [95]:
conn.close()